# Solar Power Generation Prediction using Linear Regression

## Use Case
This project aims to predict AC power output of a solar power plant using environmental and operational data such as irradiation, temperature, and time.

The goal is to help optimize energy production and improve forecasting in renewable energy systems.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import joblib

## Data Loading and Exploration

In [ ]:
df = pd.read_csv("/Users/manziivan453icloud.com/Downloads/Projects/ML_Summative_1/linear_regression_model/Plant_1_Generation_Data.csv")
df.head()

In [ ]:
print("Dataset info:")
df.info()
print("\nBasic statistics:")
df.describe()

## Data Preprocessing and Feature Engineering

In [ ]:
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
df['hour'] = df['DATE_TIME'].dt.hour
df['day'] = df['DATE_TIME'].dt.day

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

### Interpretation

From the heatmap:
- IRRADIATION has a strong positive correlation with AC_POWER
- DC_POWER is also highly correlated with AC_POWER
- Time-related features (hour) influence power generation patterns

This shows that solar power output depends heavily on sunlight intensity and time of day.

In [ ]:
# Irradiation vs Power
plt.figure()
sns.scatterplot(x=df['IRRADIATION'], y=df['AC_POWER'])
plt.title("Irradiation vs AC Power")
plt.show()

# Distribution of target
plt.figure()
sns.histplot(df['AC_POWER'], kde=True)
plt.title("Distribution of AC Power")
plt.show()

## Feature Selection

In [ ]:
features = ['IRRADIATION', 'DC_POWER', 'MODULE_TEMPERATURE', 'hour']
X = df[features]
y = df['AC_POWER']

## Train-Test Split and Standardization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Training and Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = rmse

results

In [ ]:
plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Model Comparison (RMSE)")
plt.ylabel("RMSE")
plt.show()

## Gradient Descent and Loss Curve

In [ ]:
train_losses = []
test_losses = []

sgd = SGDRegressor(max_iter=1, learning_rate='constant', eta0=0.0001, warm_start=True)

for i in range(100):
    sgd.fit(X_train_scaled, y_train)

    y_train_pred = sgd.predict(X_train_scaled)
    y_test_pred = sgd.predict(X_test_scaled)

    train_loss = mean_squared_error(y_train, y_train_pred)
    test_loss = mean_squared_error(y_test, y_test_pred)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

plt.figure()
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.legend()
plt.title("Loss Curve (Gradient Descent)")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

### Explanation

Gradient Descent is an optimization algorithm used to minimize prediction error.

The loss curve shows how the model improves over time:
- Training loss decreases as the model learns
- Test loss shows how well the model generalizes

A good model has both curves decreasing and staying close.

## Linear Regression Visualization

In [ ]:
lr_model = models["Linear Regression"]
y_pred_lr = lr_model.predict(X_test_scaled)

plt.figure()
plt.scatter(y_test, y_pred_lr)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (Linear Regression)")
plt.show()

## Save Best Model

In [ ]:
best_model = models[min(results, key=results.get)]

joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")

## Feature Importance